# 1. Imports

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
if not torch.cuda.is_available():
    print("[WARNING] No GPU detected. Go to Settings (right panel) -> Accelerator -> GPU, "
          "then restart the session, or training will be very slow.")

# 2. Configs

Structure assumed, matching your Data Explorer panel:

```
DATA_DIR/                      (e.g. .../logmel)
    logmel/                    <- 300 individual .npy files, one per song
        song1.npy
        song2.npy
        ...
    logmel_manifest.csv        <- columns: file, label (adjust FILENAME_COL/LABEL_COL below if different)
```

In [ ]:
DATA_DIR = "/kaggle/input/spectrograms-300-hcm-dataset/logmel"   # adjust to your actual path
NPY_SUBDIR = "logmel"
MANIFEST_FILE = "logmel_manifest.csv"

NPY_DIR = os.path.join(DATA_DIR, NPY_SUBDIR)
MANIFEST_PATH = os.path.join(DATA_DIR, MANIFEST_FILE)

FILENAME_COL = "file"
LABEL_COL = "label"

OUTPUT_DIR = "/kaggle/working/"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "best_model.pt")

TARGET_FRAMES = 128      # time-axis length after padding/cropping
IMG_SIZE = 224            # resnet18 pretrained input size (both mel-bin and time axes get resized to this)

ARCH = "resnet18"         # "resnet18", "resnet34", or "alexnet"
EPOCHS = 30
BATCH_SIZE = 16
LR = 1e-4
VAL_SPLIT = 0.2
FREEZE_BACKBONE = False
PATIENCE = 5               # epochs to wait for val_acc improvement before early stopping

# 3. Loading the Dataset

Unlike the per-class stacked-array format, here every song is its own `.npy` file,
so we load them individually using the manifest as the index.

In [ ]:
def load_manifest_dataset(npy_dir, manifest_path, filename_col, label_col):
    manifest = pd.read_csv(manifest_path)

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(manifest[label_col])

    X, y, filenames = [], [], []

    for fname, label in tqdm(
        zip(manifest[filename_col], y_encoded),
        total=len(manifest),
        desc="Loading spectrograms",
    ):
        fpath = fname if str(fname).endswith(".npy") else f"{fname}.npy"
        fpath = os.path.join(npy_dir, fpath)

        if not os.path.exists(fpath):
            print(f"[WARN] Missing file, skipping: {fpath}")
            continue

        arr = np.load(fpath).astype(np.float32)
        if arr.ndim != 2:
            raise ValueError(f"Expected a 2D (mel_bins, time) array, got shape {arr.shape} for {fpath}")

        X.append(arr)
        y.append(label)
        filenames.append(fname)

    print(f"Loaded {len(X)} samples across {len(label_encoder.classes_)} classes.")
    print("Classes:", list(label_encoder.classes_))

    return X, y, filenames, label_encoder


X, y, filenames, label_encoder = load_manifest_dataset(NPY_DIR, MANIFEST_PATH, FILENAME_COL, LABEL_COL)
CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES = len(CLASS_NAMES)

print("Example sample shape:", X[0].shape)

# 4. EDA

In [ ]:
counts = np.bincount(y, minlength=NUM_CLASSES)

plt.figure(figsize=(12, 4))
sns.barplot(x=CLASS_NAMES, y=counts)
plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of samples")
plt.title("Samples per class")
plt.tight_layout()
plt.show()

if counts.max() > 2 * counts.min():
    print("[NOTE] Noticeable class imbalance. Consider class weights in the loss "
          "function, or oversampling the rarer classes.")

print(f"[NOTE] {len(X)} samples across {NUM_CLASSES} classes averages to "
      f"{len(X)/NUM_CLASSES:.1f} samples/class - very small for fine-tuning a "
      f"53-layer CNN from scratch-ish. Expect heavy reliance on the pretrained "
      f"backbone + augmentation, and treat val accuracy with caution given the "
      f"tiny validation set this split will produce.")

n_show = min(6, NUM_CLASSES)
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
for i in range(n_show):
    idx = y.index(i)  # first sample belonging to class i
    axes[i].imshow(X[idx], origin="lower", aspect="auto", cmap="magma")
    axes[i].set_title(CLASS_NAMES[i])
    axes[i].set_xlabel("time frames")
    axes[i].set_ylabel("mel bins")
plt.tight_layout()
plt.show()

# 5. Preprocessing functions

In [ ]:
def standardize(spec):
    return (spec - spec.mean()) / (spec.std() + 1e-6)

def fix_length(spec, target_frames=TARGET_FRAMES):
    n_mels, n_frames = spec.shape
    if n_frames == target_frames:
        return spec
    if n_frames > target_frames:
        start = (n_frames - target_frames) // 2
        return spec[:, start:start + target_frames]
    pad_total = target_frames - n_frames
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return np.pad(spec, ((0, 0), (pad_left, pad_right)), mode="constant")

# 6. Visual sanity check: original vs. standardized + fixed-length

In [ ]:
sample = X[0]
processed = fix_length(standardize(sample))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample, origin="lower", aspect="auto", cmap="magma")
axes[0].set_title("Original")
axes[1].imshow(processed, origin="lower", aspect="auto", cmap="magma")
axes[1].set_title("Standardized + fixed-length")
plt.tight_layout()
plt.show()

# 7. Dataset class

Applies standardize + fix_length, converts to a tensor, resizes to `IMG_SIZE x IMG_SIZE`,
then repeats the single mel channel 3x so it matches resnet18's expected 3-channel
input (the source notebook skipped both of these steps, which would have crashed
on the first forward pass). SpecAugment (frequency/time masking) is applied only
on the training split.

In [ ]:
class MelSpecDataset(Dataset):
    def __init__(self, X, y, train=False):
        self.X = X
        self.y = y
        self.train = train

        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=30)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        spec = standardize(self.X[idx])
        spec = fix_length(spec)
        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)  # (1, n_mels, frames)

        # resize both axes to IMG_SIZE so it matches the pretrained resnet18 input size
        spec = F.interpolate(spec.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                              mode="bilinear", align_corners=False).squeeze(0)

        if self.train:
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)

        # repeat to 3 channels for the ImageNet-pretrained backbone
        spec = spec.repeat(3, 1, 1)

        return spec, self.y[idx]

# 8. Train-Test Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, random_state=SEED, stratify=y
)

train_ds = MelSpecDataset(X_train, y_train, train=True)
val_ds = MelSpecDataset(X_val, y_val, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_ds)} | Validation samples: {len(val_ds)}")

# 9. Model: ResNet18 (pretrained, transfer learning)

In [ ]:
def build_model(arch, num_classes):
    if arch == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "alexnet":
        model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    else:
        raise ValueError(f"Unknown arch '{arch}'")
    return model


def freeze_backbone(model, arch):
    for param in model.parameters():
        param.requires_grad = False
    if arch.startswith("resnet"):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif arch == "alexnet":
        for param in model.classifier[6].parameters():
            param.requires_grad = True
    return model


model = build_model(ARCH, NUM_CLASSES).to(DEVICE)
if FREEZE_BACKBONE:
    model = freeze_backbone(model, ARCH)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_trainable:,} / {n_total:,}")

# 10. Training loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="Training" if train else "Validation", leave=False)

    with torch.set_grad_enabled(train):
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += inputs.size(0)

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.3f}"})

    return total_loss / total, correct / total

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_acc = 0.0
counter = 0

for epoch in tqdm(range(EPOCHS), desc="Epochs"):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, DEVICE, train=False)

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
    else:
        counter += 1

    if counter >= PATIENCE:
        print("Early stopping!")
        break

print(f"\nBest validation accuracy: {best_val_acc:.3f}")
print(f"Best checkpoint saved to: {CHECKPOINT_PATH}")

# 11. Loss / accuracy curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss vs Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy vs Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

# 12. Evaluation

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()